In [ ]:
import requests
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import urllib3
import time

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
sns.set_theme(style="whitegrid")

BASE_URL = "https://localhost:7065" 
BATTLE_ENDPOINT = f"{BASE_URL}/BattleCalculator/calculate-specific-units-team"
STATS_ENDPOINT = f"{BASE_URL}/unit-stats"

UNIT_TYPES = {
    0: "Light",
    1: "Heavy",
    2: "Fast",
    3: "ShortRange",
    4: "LongRange"
}

In [ ]:
def get_current_stats():
    response = requests.get(STATS_ENDPOINT, verify=False)
    return response.json()

def update_stats(new_stats):
    requests.post(STATS_ENDPOINT, json=new_stats, verify=False)

def run_simulation(type_a, type_b, iterations=500):
    payload = {
        "battle-id": "jupyter_sim_session",
        "unit-type-A": type_a,
        "unit-type-B": type_b
    }
    
    wins_a = 0
    name_a = UNIT_TYPES[type_a]
    
    for _ in range(iterations):
        try:
            resp = requests.post(BATTLE_ENDPOINT, json=payload, verify=False)
            if resp.status_code == 200:
                if name_a in resp.json().get("winner", ""):
                    wins_a += 1
        except Exception as e:
            continue
            
    return (wins_a / iterations) * 100

In [ ]:
def generate_balance_matrix(iterations=200):
    ids = list(UNIT_TYPES.keys())
    names = [UNIT_TYPES[i] for i in ids]
    matrix = pd.DataFrame(index=names, columns=names)
    
    for id_a in ids:
        for id_b in ids:
            if id_a == id_b:
                matrix.loc[UNIT_TYPES[id_a], UNIT_TYPES[id_b]] = 50.0
                continue
            
            wr = run_simulation(id_a, id_b, iterations)
            matrix.loc[UNIT_TYPES[id_a], UNIT_TYPES[id_b]] = wr
            print(f"[{UNIT_TYPES[id_a]} vs {UNIT_TYPES[id_b]}]: {wr:.1f}% WR")
            
    return matrix.astype(float)

df_results = generate_balance_matrix(iterations=10000)

In [ ]:

def plot_balance_heatmap(df):
    plt.figure(figsize=(12, 9))
    
    cmap = sns.diverging_palette(10, 133, sep=20, as_cmap=True)
    
    ax = sns.heatmap(
        df, 
        annot=True, 
        fmt=".1f", 
        cmap="RdYlGn", 
        center=50,
        linewidths=.5,
        cbar_kws={'label': 'Win Rate (%)'},
        annot_kws={"size": 12, "weight": "bold"}
    )
    
    plt.title("Win Rate(%)", fontsize=16, pad=20, weight='bold')
    plt.xlabel("(Defender)", fontsize=12, labelpad=10)
    plt.ylabel("(Attacker)", fontsize=12, labelpad=10)
    
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.show()

def plot_average_strength(df):
    plt.figure(figsize=(12, 7))
    
    average_wr = df.mean(axis=1).sort_values()

    colors = []
    for x in average_wr:
        if x > 55 or x < 45:
            colors.append('#e74c3c') 
        else:
            colors.append('#3498db') 

   
    bars = plt.barh(average_wr.index, average_wr.values, color=colors, alpha=0.8)
    
    for bar in bars:
        width = bar.get_width()
        plt.text(
            width + 0.5, 
            bar.get_y() + bar.get_height()/2, 
            f'{width:.1f}%', 
            va='center', 
            fontsize=11, 
            weight='bold'
        )
    
    plt.axvline(x=50, color='#2ecc71', linestyle='-', linewidth=2, label='Ідеальний баланс (50%)')
    plt.axvline(x=45, color='#f39c12', linestyle='--', alpha=0.6, label='Межа допуску (45%/55%)')
    plt.axvline(x=55, color='#f39c12', linestyle='--', alpha=0.6)
    
    plt.axvspan(45, 55, color='#f1c40f', alpha=0.1)

    plt.title("Overall effectivness", fontsize=16, weight='bold', pad=20)
    plt.xlabel("Avg Win Rate (%)", fontsize=12)
    plt.xlim(0, max(average_wr.values) + 10)
    plt.legend(loc='lower right')
    
    plt.tight_layout()
    plt.show()

plot_balance_heatmap(df_results)
plot_average_strength(df_results)